# PoS tagging italiano: fine-tuning di BERT e valutazione su UD Italian-ISDT 2.15

Il compito principale è la predizione dei tag UPOS mediante fine-tuning di BERT italiano, utilizzando gli split ufficiali di UD Italian-ISDT 2.15 e parole e confini di frase gold. Il confronto esterno principale è con UDPipe 2 sul medesimo test; il report ne discute i risultati insieme alla letteratura.

Encoder congelato e decodifica CRF sono confronti di supporto. La predizione FEATS è un'estensione: ogni architettura viene prima addestrata su UPOS e poi sul compito congiunto, con tre seed, per 18 prove complessive. La selezione usa macro-F1 UPOS totale nella prima fase e la media tra macro-F1 UPOS totale e macro-F1 FEATS sulle coppie presenti nel development nella seconda.

Il [README](README.md) descrive gli artefatti finali, le cache e la verifica offline senza training o inferenza. Per consultare i risultati sono sufficienti il [report](report/report.pdf) e gli output salvati; l'esecuzione completa del notebook esegue le prove mancanti e rivaluta i modelli.

## 1. Environment


In [ ]:
%pip install -q "transformers==4.46.3" "datasets==3.1.0" "accelerate==1.1.1" "scikit-learn==1.8.0" "pandas>=2,<4" "conllu==6.0.0" "pytorch-crf==0.7.2"


## 2. Configuration


In [ ]:
import gc, hashlib, json, platform
from collections import Counter
from importlib.metadata import version
from pathlib import Path
from typing import Any, cast
from urllib.request import urlopen
import numpy as np
import pandas as pd
# Load pandas/PyArrow before PyTorch to avoid a Windows native import conflict.
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset as TorchDataset
from conllu import parse_incr
from datasets import Dataset
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torchcrf import CRF
from transformers import (AutoTokenizer, BertConfig, BertModel, BertPreTrainedModel,
                          Trainer, TrainingArguments, set_seed)
from transformers.modeling_outputs import TokenClassifierOutput

SEEDS = [42, 123, 456]
MODEL_NAME = "dbmdz/bert-base-italian-cased"
MODEL_REVISION = "843e404d71f4c63972443e3463456f59024db278"
UD_VERSION = "r2.15"
MAX_LENGTH, EPOCHS = 512, 5
FEATS_WEIGHT = 1.0
NONE = "<NONE>"
PROJECT = Path.cwd()
if (PROJECT / "natural-language-processing").is_dir():
    PROJECT /= "natural-language-processing"
UPOS_DIR = PROJECT / "outputs/bert_upos_then_feats_v1"
EXPERIMENT_DIR = PROJECT / "outputs/bert_upos_then_feats_v2"
SELECTION_METRICS = {"upos": "upos_macro_f1_all", "upos_feats": "joint_macro_f1"}
ARCHITECTURES = {
    "frozen": {"name": "Frozen BERT", "learning_rate": 1e-3, "crf": False},
    "linear": {"name": "Fine-tuned BERT", "learning_rate": 2e-5, "crf": False},
    "crf": {"name": "Fine-tuned BERT + UPOS CRF", "learning_rate": 2e-5, "crf": True},
}
# Stage 1: UPOS only. Stage 2: continue the matching UPOS model with FEATS heads.
STAGES = {"upos": "UPOS", "upos_feats": "UPOS + FEATS"}
SYSTEMS = {f"{architecture}_{stage}": {
    **options, "name": f"{options['name']} / {task}", "architecture": architecture, "stage": stage,
    "parent": f"{architecture}_upos" if stage == "upos_feats" else None,
} for stage, task in STAGES.items() for architecture, options in ARCHITECTURES.items()}
ACTIVE_SYSTEMS = list(SYSTEMS)
print("Device:", torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")


## 3. Data


In [ ]:
def canonical_feats(features):
    return {key: ",".join(sorted(value.split(","))) for key, value in (features or {}).items()}

sentences, file_hashes = {}, {}
data_dir = PROJECT / "data" / f"UD_Italian-ISDT-{UD_VERSION}"
data_dir.mkdir(parents=True, exist_ok=True)
for split in ("train", "dev", "test"):
    path = data_dir / f"it_isdt-ud-{split}.conllu"
    if not path.exists():
        url = f"https://raw.githubusercontent.com/UniversalDependencies/UD_Italian-ISDT/{UD_VERSION}/{path.name}"
        with urlopen(url, timeout=120) as response:
            content = response.read()
        assert content and b"# sent_id" in content, "Invalid dataset download"
        path.write_bytes(content)
    file_hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    with path.open(encoding="utf-8") as stream:
        words = [[word for word in sentence if isinstance(word["id"], int)] for sentence in parse_incr(stream)]
    sentences[split] = [{"tokens": [w["form"] for w in row], "upos": [w["upos"] for w in row],
                         "feats": [canonical_feats(w["feats"]) for w in row]} for row in words if row]

train = sentences["train"]
label_names = sorted({tag for row in train for tag in row["upos"]})
feature_names = sorted({key for row in train for feats in row["feats"] for key in feats})
head_names = ["UPOS", *feature_names]
head_labels = {"UPOS": label_names, **{key: [NONE, *sorted({feats[key] for row in train
    for feats in row["feats"] if key in feats})] for key in feature_names}}
label_ids = {head: {label: i for i, label in enumerate(values)} for head, values in head_labels.items()}
training_pairs = {f"{key}={value}" for key in feature_names for value in head_labels[key] if value != NONE}

display(pd.DataFrame([{"Split": split, "Sentences": len(rows), "Words": sum(len(r["tokens"]) for r in rows)}
                      for split, rows in sentences.items()]))
display(pd.DataFrame([{"Head": head, "Classes": len(values), "Labels": ", ".join(values)}
                      for head, values in head_labels.items()]))
for split in ("dev", "test"):
    unseen = sorted({f"{key}={value}" for row in sentences[split] for feats in row["feats"]
                     for key, value in feats.items()} - training_pairs)
    print(f"{split}: FEATS pairs absent from training: {unseen}")


## 4. Tokenization


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, use_fast=True)
assert tokenizer.is_fast and tokenizer.padding_side == "right"
assert 0 < MAX_LENGTH <= tokenizer.model_max_length

def prepare_data(examples):
    sequences, positions, lengths = [], [], []
    for sentence_index, row in enumerate(examples):
        sequence = tokenizer(row["tokens"], is_split_into_words=True, truncation=False)
        lengths.append(len(sequence["input_ids"]))
        if lengths[-1] > MAX_LENGTH:
            raise ValueError(f"Sentence {sentence_index} exceeds {MAX_LENGTH} tokens")
        labels, covered, previous = [], [], None
        for word_index in sequence.word_ids():
            values = [-100] * len(head_names)
            if word_index is not None and word_index != previous:
                values = [label_ids["UPOS"][row["upos"][word_index]], *[
                    label_ids[key].get(row["feats"][word_index].get(key, NONE), 0) for key in feature_names]]
                covered.append(word_index)
            labels.append(values)
            previous = word_index
        assert covered == list(range(len(row["tokens"])))
        sequences.append(dict(sequence, labels=labels))
        positions.extend((sentence_index, word_index) for word_index in covered)
    return Dataset.from_list(sequences), positions, {"sentences": len(examples), "words": len(positions),
        "longest_sentence_tokens": max(lengths, default=0)}

prepared = {split: prepare_data(rows) for split, rows in sentences.items()}
model_data = {split: values[0] for split, values in prepared.items()}

def collate(features):
    labels = [torch.tensor(row["labels"], dtype=torch.long) for row in features]
    batch = tokenizer.pad([{key: value for key, value in row.items() if key != "labels"}
                           for row in features], padding=True, return_tensors="pt")
    batch["labels"] = pad_sequence(labels, batch_first=True, padding_value=-100)
    batch["word_mask"] = batch["labels"][:, :, 0] != -100
    return batch

print("Alignment verified: one label vector per word; no truncation.")


## 5. Metrics


In [ ]:
metric_names = ["upos_accuracy", "upos_macro_f1_all", "upos_macro_f1_present", "feats_exact_accuracy",
                "feats_attribute_accuracy", "feats_macro_f1_all", "feats_macro_f1_present", "joint_accuracy"]
def gold_annotations(split):
    rows = sentences[split]
    return ([rows[s]["upos"][w] for s, w in prepared[split][1]],
            [rows[s]["feats"][w] for s, w in prepared[split][1]])

def decode_output(predictions, label_tensor, task_heads):
    if not isinstance(label_tensor, np.ndarray):
        raise ValueError("Expected gold label tensors from Trainer")
    values = predictions[0] if isinstance(predictions, tuple) else predictions
    if not isinstance(values, np.ndarray):
        raise ValueError("Expected prediction arrays from Trainer")
    if values.ndim == 4:
        values = values.argmax(-1)
    assert values.shape[:2] == label_tensor.shape[:2] and values.shape[2] == len(task_heads)
    valid = label_tensor[:, :, 0] != -100
    ids = values[valid]
    upos = [head_labels["UPOS"][int(row[0])] for row in ids]
    feats = [{key: head_labels[key][int(row[i])] for i, key in enumerate(task_heads[1:], 1)
              if int(row[i]) != 0} for row in ids]
    return upos, feats if len(task_heads) > 1 else None

def feature_sets(rows):
    return [set(f"{key}={value}" for key, value in feats.items()) for feats in rows]

def feature_scores(gold, predicted):
    gold_sets, pred_sets = feature_sets(gold), feature_sets(predicted)
    support = Counter(pair for row in gold_sets for pair in row)
    guesses = Counter(pair for row in pred_sets for pair in row)
    matches = Counter(pair for a, b in zip(gold_sets, pred_sets) for pair in a & b)
    rows = []
    for pair in sorted(training_pairs | set(support) | set(guesses)):
        tp, actual, guessed = matches[pair], support[pair], guesses[pair]
        rows.append({"feature": pair, "precision": tp / guessed if guessed else 0.,
            "recall": tp / actual if actual else 0., "f1": 2*tp/(actual+guessed) if actual+guessed else 0.,
            "support": actual, "tp": tp, "fp": guessed-tp, "fn": actual-tp})
    return pd.DataFrame(rows)

def score_predictions(gold_upos, gold_feats, pred_upos, pred_feats):
    assert len(gold_upos) == len(pred_upos) == len(gold_feats)
    metrics = {
        "upos_accuracy": float(accuracy_score(gold_upos, pred_upos)),
        "upos_macro_f1_all": float(f1_score(gold_upos, pred_upos, labels=label_names, average="macro", zero_division=0)),
        "upos_macro_f1_present": float(f1_score(gold_upos, pred_upos, labels=sorted(set(gold_upos)), average="macro", zero_division=0)),
    }
    if pred_feats is not None:
        assert len(gold_feats) == len(pred_feats)
        table = feature_scores(gold_feats, pred_feats)
        exact = np.array([a == b for a, b in zip(gold_feats, pred_feats)])
        attributes = sorted(set(feature_names) | {key for row in gold_feats + pred_feats for key in row})
        metrics.update({
            "feats_exact_accuracy": float(exact.mean()),
            "feats_attribute_accuracy": float(np.mean([a.get(key, NONE) == b.get(key, NONE)
                for a, b in zip(gold_feats, pred_feats) for key in attributes])),
            "feats_macro_f1_all": float(table.f1.mean()),
            "feats_macro_f1_present": float(table.loc[table.support > 0, "f1"].mean()),
            "joint_accuracy": float(np.mean(exact & (np.array(gold_upos) == np.array(pred_upos)))),
        })
    return metrics

def metrics_for(split, task_heads):
    gold_upos, gold_feats = gold_annotations(split)
    def compute(output):
        pred_upos, pred_feats = decode_output(output.predictions, output.label_ids, task_heads)
        valid = output.label_ids[:, :, 0] != -100
        expected = np.array([label_ids["UPOS"][tag] for tag in gold_upos])
        assert np.array_equal(output.label_ids[:, :, 0][valid], expected)
        scores = score_predictions(gold_upos, gold_feats, pred_upos, pred_feats)
        if split == "dev" and pred_feats is not None:
            scores["joint_macro_f1"] = (scores["upos_macro_f1_all"] + scores["feats_macro_f1_present"]) / 2
        return scores
    return compute


def reduce_logits(logits, labels):
    logits = logits[0] if isinstance(logits, tuple) else logits
    return logits.argmax(-1)  # Store word/head IDs during evaluation, not full score tensors.


## 6. Models


In [ ]:
class JointBert(BertPreTrainedModel):
    pipeline_version = "staged_upos_feats_v1"
    crf_initialization = "uniform_v1"

    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.heads = nn.ModuleDict({key: nn.Linear(config.hidden_size, len(config.head_labels[key]))
                                    for key in config.head_names})
        self.crfs = nn.ModuleDict({key: CRF(len(config.head_labels[key]), batch_first=True)
                                   for key in config.crf_heads})
        self.post_init()

    def _init_weights(self, module):
        super()._init_weights(module)
        if isinstance(module, CRF):
            # HF fast loading skips constructor initialization for new parameters.
            module.reset_parameters()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                labels=None, word_mask=None) -> TokenClassifierOutput:
        if word_mask is None or not bool(word_mask.any(dim=1).all()):
            raise ValueError("Every sentence needs a first-subword mask with at least one word")
        hidden = self.dropout(self.bert(input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids, return_dict=True).last_hidden_state)
        lengths = word_mask.sum(1)
        sequence_mask = torch.arange(int(lengths.max()), device=hidden.device)[None, :] < lengths[:, None]
        width = max(len(values) for values in self.config.head_labels.values())
        losses, scores = [], []
        for index, key in enumerate(self.config.head_names):
            emissions = self.heads[key](hidden)
            if key in self.crfs:
                crf = cast(CRF, self.crfs[key])
                words = pad_sequence([row[mask] for row, mask in zip(emissions, word_mask)], batch_first=True).float()
                if labels is not None:
                    tags = pad_sequence([row[mask, index] for row, mask in zip(labels, word_mask)], batch_first=True)
                    losses.append(-crf(words, tags, mask=sequence_mask, reduction="token_mean"))
                if not self.training:
                    paths = crf.decode(words, mask=sequence_mask)
                    emissions = torch.zeros_like(emissions)
                    for row, path in enumerate(paths):
                        emissions[row, word_mask[row], torch.tensor(path, device=hidden.device)] = 1
            elif labels is not None:
                losses.append(nn.functional.cross_entropy(emissions.reshape(-1, emissions.size(-1)),
                    labels[:, :, index].reshape(-1), ignore_index=-100))
            scores.append(nn.functional.pad(emissions, (0, width-emissions.size(-1)), value=-1e4))
        loss = losses[0] if losses else None
        if len(losses) > 1:
            loss = loss + self.config.feats_weight * torch.stack(losses[1:]).mean()
        if loss is not None and not bool(torch.isfinite(loss)):
            raise FloatingPointError("Non-finite loss: stop and check model initialization")
        return TokenClassifierOutput(loss=loss, logits=cast(torch.FloatTensor, torch.stack(scores, dim=2)))

def make_model(system, checkpoint: str | Path | None = None) -> JointBert:
    spec = SYSTEMS[system]
    if checkpoint is None and spec["parent"] is not None:
        raise ValueError("The UPOS + FEATS stage requires its matching UPOS checkpoint")
    source = str(checkpoint) if checkpoint is not None else MODEL_NAME
    options: dict[str, Any] = {} if checkpoint is not None else {"revision": MODEL_REVISION}
    config = BertConfig.from_pretrained(source, **options)
    if checkpoint is not None:
        assert config.pipeline_version == JointBert.pipeline_version
        assert config.architecture == spec["architecture"]
        assert config.task_stage == spec["stage"] or (config.task_stage == "upos" and spec["stage"] == "upos_feats")
        assert config.head_labels == {key: head_labels[key] for key in config.head_names}
        if config.crf_heads and config.crf_initialization != JointBert.crf_initialization:
            raise ValueError("Incompatible CRF initialization")
    config.head_names = ["UPOS"] if spec["stage"] == "upos" else head_names
    config.head_labels = {key: head_labels[key] for key in config.head_names}
    config.crf_heads = ["UPOS"] if spec["crf"] else []
    config.feats_weight = FEATS_WEIGHT if spec["stage"] == "upos_feats" else 0.
    config.architecture, config.task_stage = spec["architecture"], spec["stage"]
    config.pipeline_version, config.crf_initialization = JointBert.pipeline_version, JointBert.crf_initialization
    model = cast(JointBert, JointBert.from_pretrained(source, config=config, **options))
    if any(not bool(torch.isfinite(p).all()) or (checkpoint is None and float(p.detach().abs().max()) > 0.1)
           for p in model.crfs.parameters()):
        raise RuntimeError("Invalid CRF transitions; training has not started")
    if spec["architecture"] == "frozen":
        model.bert.requires_grad_(False)
    return model


def make_trainer(system, model, run_dir, seed, training=False):
    spec = SYSTEMS[system]
    options: dict[str, Any] = dict(num_train_epochs=EPOCHS, learning_rate=spec["learning_rate"],
        per_device_train_batch_size=8, weight_decay=0.01, warmup_ratio=0.1,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model=SELECTION_METRICS[spec["stage"]], greater_is_better=True,
        save_total_limit=1 if spec["stage"] == "upos" else None,
        save_only_model=spec["stage"] == "upos_feats") if training else {}
    args = TrainingArguments(output_dir=str(run_dir / ("checkpoints" if training else "evaluation")),
        per_device_eval_batch_size=16, seed=seed, data_seed=seed,
        fp16=torch.cuda.is_available(), report_to="none", label_names=["labels"], **options)
    return Trainer(model=model, args=args, processing_class=tokenizer, data_collator=collate,
        train_dataset=cast(TorchDataset[Any], model_data["train"]) if training else None,
        eval_dataset=cast(TorchDataset[Any], model_data["dev"]) if training else None,
        compute_metrics=metrics_for("dev" if training else "test", model.config.head_names), preprocess_logits_for_metrics=reduce_logits)

def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 7. Training


In [ ]:
if getattr(JointBert, "pipeline_version", None) != "staged_upos_feats_v1":
    raise RuntimeError("Outdated model definition: restart the kernel and run all cells in order")

experiment = {
    "schema": JointBert.pipeline_version, "crf_initialization": JointBert.crf_initialization,
    "model": MODEL_NAME, "revision": MODEL_REVISION,
    "data_hashes": file_hashes, "head_labels": head_labels, "head_names": head_names,
    "systems": SYSTEMS, "seeds": SEEDS, "epochs_per_stage": EPOCHS, "max_length": MAX_LENGTH,
    "feats_weight": FEATS_WEIGHT, "batch_size_train": 8, "batch_size_eval": 16,
    "weight_decay": 0.01, "warmup_ratio": 0.1, "fp16": torch.cuda.is_available(),
    "selection_metric": SELECTION_METRICS, "save_total_limit": None, "save_only_model": True,
    "preprocessing": {split: values[2] for split, values in prepared.items()},
    "python": platform.python_version(), "packages": {name: version(name) for name in
        ["torch", "transformers", "datasets", "accelerate", "scikit-learn", "numpy", "pandas", "conllu", "pytorch-crf"]},
}

def register_experiment(root, settings, legacy=None):
    path = root / "experiment.json"
    if path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))
        assert saved == settings or (legacy is not None and saved == legacy), f"Configuration mismatch: {root}"
        return saved
    root.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(settings, indent=2), encoding="utf-8")
    return settings

legacy_upos = {key: value for key, value in experiment.items()
               if key not in {"selection_metric", "save_total_limit", "save_only_model"}}
legacy_upos["selection_metric"] = "development UPOS macro-F1 all (both stages)"
upos_experiment = {**experiment, "selection_metric": {"upos": SELECTION_METRICS["upos"]},
                   "save_total_limit": 1, "save_only_model": False}
upos_experiment = register_experiment(UPOS_DIR, upos_experiment, legacy_upos)
experiment = register_experiment(EXPERIMENT_DIR, experiment)

def run_directory(system, seed):
    root = UPOS_DIR if SYSTEMS[system]["stage"] == "upos" else EXPERIMENT_DIR
    return root / system / f"seed_{seed}"

def saved_run_info(system, seed):
    run_dir = run_directory(system, seed)
    info = json.loads((run_dir / "training_info.json").read_text(encoding="utf-8"))
    expected = upos_experiment if SYSTEMS[system]["stage"] == "upos" else experiment
    assert info["experiment"] == expected and info["system"] == system and info["seed"] == seed
    assert (run_dir / "best_model/model.safetensors").exists(), "Missing saved model"
    return info

def train_run(system, seed):
    spec = SYSTEMS[system]
    run_dir = run_directory(system, seed)
    info_path, model_path = run_dir / "training_info.json", run_dir / "best_model"
    source, source_hash = None, None
    if spec["parent"] is not None:
        saved_run_info(spec["parent"], seed)
        source = run_directory(spec["parent"], seed) / "best_model"
        with (source / "model.safetensors").open("rb") as stream:
            source_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    metric = SELECTION_METRICS[spec["stage"]]
    if info_path.exists() != (model_path / "model.safetensors").exists():
        raise RuntimeError(f"Incomplete saved run: {run_dir}. Restore its files or choose a new output directory.")
    if info_path.exists():
        info = saved_run_info(system, seed)
        assert info["training_args"]["metric_for_best_model"] == metric
        assert info["source_sha256"] == source_hash, "The parent checkpoint changed: choose a new EXPERIMENT_DIR"
        print(f"Reusing {system}, seed {seed}")
        return
    set_seed(seed)
    model = make_model(system, source)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if spec["architecture"] == "frozen":
        assert trainable == sum(p.numel() for p in model.heads.parameters())
    trainer = make_trainer(system, model, run_dir, seed, training=True)
    assert trainer.args.metric_for_best_model == metric, "Restart the kernel and run all cells in order"
    print(f"{spec['name']} | seed {seed}; trainable parameters: {trainable:,}; selection: {metric}")
    trainer.train()
    best_metric = trainer.state.best_metric
    if best_metric is None or not np.isfinite(best_metric):
        raise RuntimeError("Training finished without a finite development selection score")
    trainer.save_model(str(model_path))
    pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "training_history.csv", index=False)
    info = {"system": system, "seed": seed, "experiment": upos_experiment if spec["stage"] == "upos" else experiment,
        "trainable_parameters": trainable, "selection_metric": metric, "best_dev_score": float(best_metric),
        "best_checkpoint": trainer.state.best_model_checkpoint, "source_checkpoint": str(source) if source is not None else MODEL_NAME,
        "source_sha256": source_hash, "training_args": trainer.args.to_dict()}
    info_path.write_text(json.dumps(info, indent=2, default=str), encoding="utf-8")
    del trainer, model
    release_memory()

completed = {(system, seed): all(path.exists() for path in
    [run_directory(system, seed) / "training_info.json", run_directory(system, seed) / "best_model/model.safetensors"])
    for system in ACTIVE_SYSTEMS for seed in SEEDS}
plan_path = EXPERIMENT_DIR / "run_plan.json"
if not plan_path.exists():
    plan_path.write_text(json.dumps({"active_systems": ACTIVE_SYSTEMS, "seeds": SEEDS,
        "reused_upos_runs": sum(done for (system, seed), done in completed.items() if SYSTEMS[system]["stage"] == "upos"),
        "training_runs": sum(not done for done in completed.values())}, indent=2), encoding="utf-8")
print(f"{sum(completed.values())} saved runs; {sum(not done for done in completed.values())} missing runs.")
for system in ACTIVE_SYSTEMS:
    for seed in SEEDS:
        train_run(system, seed)


## 8. Evaluation


In [ ]:
def feats_text(features):
    return "|".join(f"{key}={value}" for key, value in sorted(features.items())) or "_"

def attribute_scores(gold, predicted):
    rows = []
    for key in sorted(set(feature_names) | {k for row in gold + predicted for k in row}):
        actual = [row.get(key, NONE) for row in gold]
        guessed = [row.get(key, NONE) for row in predicted]
        labels = sorted(set(head_labels.get(key, [NONE])) | set(actual) | set(guessed))
        rows.append({"attribute": key, "accuracy": accuracy_score(actual, guessed),
            "macro_f1_all": f1_score(actual, guessed, labels=labels, average="macro", zero_division=0),
            "macro_f1_present": f1_score(actual, guessed, labels=sorted(set(actual)), average="macro", zero_division=0),
            "gold_nonempty": sum(value != NONE for value in actual)})
    return pd.DataFrame(rows)

for seed in SEEDS:
    for system in ACTIVE_SYSTEMS:
        assert (run_directory(system, seed) / "training_info.json").exists(), f"Complete all {len(SEEDS) * len(ACTIVE_SYSTEMS)} active training runs first"

gold_upos, gold_feats = gold_annotations("test")
positions = prepared["test"][1]
run_results, prediction_frames = [], {}
for seed in SEEDS:
    for system in ACTIVE_SYSTEMS:
        run_dir = run_directory(system, seed)
        saved_run_info(system, seed)
        model = make_model(system, run_dir / "best_model")
        expected_heads = ["UPOS"] if SYSTEMS[system]["stage"] == "upos" else head_names
        assert model.config.head_names == expected_heads
        assert model.config.head_labels == {key: head_labels[key] for key in expected_heads}
        trainer = make_trainer(system, model, run_dir, seed)
        output = trainer.predict(cast(TorchDataset[Any], model_data["test"]))
        assert output.metrics is not None, "Missing test metrics"
        pred_upos, pred_feats = decode_output(output.predictions, output.label_ids, model.config.head_names)
        metric_keys = metric_names if pred_feats is not None else metric_names[:3]
        scores = {"seed": seed, "system": system, **{
            key: float(output.metrics[f"test_{key}"]) for key in metric_keys}}
        frame = pd.DataFrame({"seed": seed, "sentence_index": [s for s, w in positions],
            "word_index": [w for s, w in positions], "word": [sentences["test"][s]["tokens"][w] for s, w in positions],
            "true_label": gold_upos, "predicted_label": pred_upos,
            "true_feats": [feats_text(row) for row in gold_feats], "predicted_feats": [feats_text(row) for row in pred_feats] if pred_feats is not None else None})
        frame.to_csv(run_dir / "predictions.csv", index=False)
        wrong = frame.true_label != frame.predicted_label
        if pred_feats is not None:
            wrong = wrong | (frame.true_feats != frame.predicted_feats)
        frame[wrong].to_csv(run_dir / "errors.csv", index=False)
        (run_dir / "test_results.json").write_text(json.dumps(scores, indent=2), encoding="utf-8")
        report = classification_report(gold_upos, pred_upos, labels=label_names, target_names=label_names, output_dict=True, zero_division=0)
        pd.DataFrame(report).T.loc[label_names].to_csv(run_dir / "per_tag_metrics.csv")
        if pred_feats is not None:
            feature_scores(gold_feats, pred_feats).to_csv(run_dir / "per_feature_metrics.csv", index=False)
            attribute_scores(gold_feats, pred_feats).to_csv(run_dir / "per_attribute_metrics.csv", index=False)
        run_results.append(scores)
        prediction_frames[(seed, system)] = frame
        del trainer, model, output
        release_memory()
results = pd.DataFrame(run_results)
results.to_csv(EXPERIMENT_DIR / "results.csv", index=False)


## 9. Comparison

Il riepilogo `outputs/bert_upos_then_feats_v2/comparison.csv` contiene le sei configurazioni BERT; **la riga UDPipe in questo file riguarda soltanto UPOS**. Il confronto esterno morfologico e l'accuracy congiunta del report vengono riprodotti separatamente dalle risposte UDPipe e dalle predizioni BERT già salvate.

Dalla cartella `natural-language-processing/`, con `data/` e `outputs/` disponibili, eseguire:

```console
python -B evaluate_udpipe_morphology.py
```

Lo script non richiede rete, pesi dei modelli, training o inferenza. Verifica dati e allineamento e aggiorna soltanto:

- [`morphology_comparison.csv`](outputs/references/udpipe_2_15/morphology_comparison.csv): confronto BERT/UDPipe, in percentuale;
- [`morphology_per_feature_metrics.csv`](outputs/references/udpipe_2_15/morphology_per_feature_metrics.csv): metriche per coppia FEATS, in frazioni;
- [`morphology_evaluation.json`](outputs/references/udpipe_2_15/morphology_evaluation.json): metriche in frazioni, controlli e provenienza.

Queste metriche usano gli insiemi FEATS completi e la correttezza congiunta UPOS+FEATS; sono distinte dai punteggi ufficiali UDPipe `UFeats` e `AllTags`.


In [ ]:
summary = results.groupby("system")[metric_names].agg(["mean", "std"]).reindex(ACTIVE_SYSTEMS) * 100
print("Mean and sample standard deviation (%); FEATS metrics apply to the second stage only.")
display(summary.round(3))
summary.to_csv(EXPERIMENT_DIR / "comparison_mean_std.csv")

comparison = results.groupby("system")[metric_names].mean().reindex(ACTIVE_SYSTEMS).reset_index()
comparison["system"] = comparison["system"].map({key: value["name"] for key, value in SYSTEMS.items()})
reference_dir = PROJECT / "outputs/references/udpipe_2_15"
if (reference_dir / "predictions.csv").exists():
    info = json.loads((reference_dir / "reference_info.json").read_text(encoding="utf-8"))
    assert info["model"] == "italian-isdt-ud-2.15-241121" and info["gold_annotations_sent"] is False
    assert info["test_sha256"] == file_hashes["it_isdt-ud-test.conllu"]
    reference = pd.read_csv(reference_dir / "predictions.csv", keep_default_na=False)
    expected = prediction_frames[(SEEDS[0], "linear_upos")]
    columns = ["sentence_index", "word_index", "word", "true_label"]
    assert reference[columns].equals(expected[columns])
    guessed = reference.predicted_label.tolist()
    reference_scores = {"system": "UDPipe 2.15", **score_predictions(gold_upos, gold_feats, guessed, None)}
    comparison = pd.concat([comparison, pd.DataFrame([reference_scores])], ignore_index=True)
comparison.to_csv(EXPERIMENT_DIR / "comparison.csv", index=False)
display(comparison.set_index("system").mul(100).round(3).fillna("—"))

paired = results.pivot(index="seed", columns="system", values=metric_names).reindex(columns=ACTIVE_SYSTEMS, level="system") * 100
paired.to_csv(EXPERIMENT_DIR / "paired_metrics.csv")
display(paired.round(3))

# Paired change after adding FEATS and continuing training (percentage points).
stage_changes = []
for architecture in ARCHITECTURES:
    before = results.loc[results.system == f"{architecture}_upos"].set_index("seed")
    after = results.loc[results.system == f"{architecture}_upos_feats"].set_index("seed")
    change = (after[metric_names[:3]] - before[metric_names[:3]]) * 100
    change["architecture"] = architecture
    stage_changes.append(change.reset_index())
stage_deltas = pd.concat(stage_changes, ignore_index=True)
stage_deltas.to_csv(EXPERIMENT_DIR / "upos_stage_changes.csv", index=False)
display(stage_deltas.round(3))


## 10. Errors


In [ ]:
error_rows = []
for system in ACTIVE_SYSTEMS:
    counts = Counter()
    for seed in SEEDS:
        frame = prediction_frames[(seed, system)]
        wrong = frame[frame.true_label != frame.predicted_label]
        counts.update(zip(wrong.true_label, wrong.predicted_label))
    error_rows.extend({"system": system, "true_tag": a, "predicted_tag": b, "errors": count}
                      for (a, b), count in counts.most_common(3))
error_table = pd.DataFrame(error_rows)
print("Top three UPOS confusions per system, aggregated across seeds.")
display(error_table)
error_table.to_csv(EXPERIMENT_DIR / "error_summary.csv", index=False)

feature_errors = []
for system in ACTIVE_SYSTEMS:
    if SYSTEMS[system]["stage"] != "upos_feats":
        continue
    rows = [pd.read_csv(run_directory(system, seed) / "per_feature_metrics.csv") for seed in SEEDS]
    table = pd.concat(rows).groupby("feature")[["fp", "fn"]].sum()
    table["errors"] = table.fp + table.fn
    table["system"] = system
    feature_errors.append(table.nlargest(5, "errors").reset_index())
feature_error_table = pd.concat(feature_errors, ignore_index=True)
print("Top five FEATS pair errors per system (false positives + false negatives across seeds).")
display(feature_error_table)
feature_error_table.to_csv(EXPERIMENT_DIR / "feature_error_summary.csv", index=False)
